In [53]:
## import libraries
import pandas as pd

In [55]:
## import data
assessed_df = pd.read_excel('input/1_AC Property Assessments_20260714.xlsx',dtype={'PROPERTYHOUSENUM':object,
'PROPERTYFRACTION':object,
'PROPERTYZIP':object,
'MUNICODE':object,
'TAXYEAR':object,
'SCHOOLCODE':object,
'NEIGHCODE':object,
'TAXCODE':object,
'OWNERCODE':object,
'USECODE':object,
'SALECODE':object,
})

## Clean data

In [56]:
## change column names to lowercase
assessed_df.columns = assessed_df.columns.str.lower()

In [57]:
## change sale date to datetime datatype
assessed_df['saledate'] = assessed_df['saledate'].astype('datetime64[ns]')

In [58]:
## create a new column for sale year
assessed_df['sale_yr']=assessed_df['saledate'].dt.year

In [59]:
## strip muni names of whitespace
assessed_df['munidesc'] = assessed_df['munidesc'].str.strip()                           

In [60]:
## check number of rows in original dataset
len(assessed_df)

584925

## Filtering for recent, valid, taxable sales

Per advice given to me by Dom Gambino, former manager of the Office of Property Assessments, I thought to limit the dataset to 2025 and 2026 to avoid a high frequency of properties where values have been appealed. I'm inclined to stick with Gambino's recommendation because the median ratio of the sample ends up being closer to STEB's determined CLR with his approach (as opposed to creating a sample spread across a longer timeframe).

In [111]:
## filter data for sales completed between 2024 and 2026
sales_25_26 = assessed_df[(assessed_df['sale_yr'] >= 2025) & (assessed_df['sale_yr'] <= 2026)]

In [112]:
## checking the length of the filtered dataset
len(sales_25_26)

41537

In [113]:
## filter for taxable parcels
taxable_25_26 = sales_25_26[sales_25_26['taxcode'] == 'T']

In [114]:
## check new length after filtering for taxable parcels
len(taxable_25_26)

40870

In [115]:
## filter for valid sales
valid_25_26 = taxable_25_26[taxable_25_26['salecode'].isin([0,'U','UR'])]

No parcels with sale codes of 'U' or 'UR' are included in the dataset. I learned this in hindsight after trying to filter for them. The Allegheny County OPA considers those two, as well as valid sales (coded as 0).

In [116]:
## check the new length
len(valid_25_26)

12574

The International Association of Assessing Officers recommends having at least 15 sales to observe per sample. Other experts, such as CMU Professor Robert Strauss, suggest including at least 30 sales per sample. 

I'm going to stick with IAAO's recommendation, mostly because I want to analyze the data at the municipal level rather than the school district level, which would yield more samples per district, but less granularity. 

In [120]:
## grouping the latest dataset according to municipality, then sorting from greatest number of sales to least.
## looking at the tail-end of the data to see which munis have fewer than 15 sales
not_observable = valid_25_26['munidesc'].value_counts().tail(47).reset_index()
not_observable

,munidesc,count
0,Oakdale,14
1,Ben Avon,14
2,Homestead,14
3,Whitaker,14
4,Sharpsburg,13
5,1st Ward - CLAIRTON,13
6,Bell Acres,13
7,Fawn,13
8,5th Ward - PITTSBURGH,12
9,Elizabeth Boro,12


In [121]:
## make a list of just the muni names
not_obv_list = not_observable['munidesc'].tolist()

In [122]:
## choosing all munis EXCEPT the ones listed above
valid_df = valid_25_26[~valid_25_26['munidesc'].isin(not_obv_list)]

In [123]:
## look at length of the dataset
len(valid_df)

12185

## Which homes have the greatest difference between their property assessments and sales prices?

In [124]:
## create a new column that equals the ratio of assessed values to sales prices
valid_df['ratio'] = valid_df['fairmarkettotal']/valid_df['saleprice']

/var/folders/09/dth632p523n1dk0qt7d74bsr0000gn/T/ipykernel_3690/2953436989.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_df['ratio'] = valid_df['fairmarkettotal']/valid_df['saleprice']


This median sales ratio for this sample is relatively close to [STEB's determination](https://dced.pa.gov/download/common-level-ratio-2025/?wpdmdl=129658&refresh=6a0dccda0a7f91779289306) of the county's CLR, which is 49.3 for 2025. 

In [125]:
## find the median ratio for the entire sample
sample_ratio = valid_df['ratio'].median()
sample_ratio

np.float64(0.49945205479452054)

In [126]:
## find the median sales ratio per municipality
muni_median = valid_df.groupby('munidesc')['ratio'].median().reset_index()

From IAAO's [guidelines on sales ratio studies](https://www.iaao.org/wp-content/uploads/StandardonRatioStudies_Exposure-Mar2026.pdf): "While the target level of valuation should be 1.00, a valuation level between 0.90 and 1.10 is considered acceptable for any class of property."

In [127]:
## sort the median ratios highest to lowest
muni_median.sort_values(by='ratio',ascending=False).head(20)

,munidesc,ratio
19,2nd Ward - PITTSBURGH,0.626891
7,17th Ward - PITTSBURGH,0.622765
45,Cheswick,0.580051
59,Findlay,0.577415
47,Collier,0.577406
73,Leet,0.562541
92,Pine,0.555620
91,Pennsbury Village,0.552469
85,North Fayette,0.551026
107,Springdale Twp,0.549231


In [128]:
## create a new column that is the ratio of the muni's ratio to the county's
muni_median['sample_comparison'] = muni_median['ratio']/sample_ratio

In [130]:
muni_median.sort_values(by='sample_comparison',ascending=False)

,munidesc,ratio,sample_comparison
19,2nd Ward - PITTSBURGH,0.626891,1.255157
7,17th Ward - PITTSBURGH,0.622765,1.246897
45,Cheswick,0.580051,1.161375
59,Findlay,0.577415,1.156097
47,Collier,0.577406,1.156079
...,...,...,...
0,10th Ward - PITTSBURGH,0.376695,0.754216
26,7th Ward - McKEESPORT,0.375999,0.752824
3,12th Ward - PITTSBURGH,0.366489,0.733783
8,18th Ward - PITTSBURGH,0.365890,0.732582


In [131]:
## save to a csv
muni_ratio_comparison = muni_median.sort_values(by='sample_comparison',ascending=False)
muni_ratio_comparison.to_csv('output/muni_ratio_comparison.csv')

In [132]:
## defining a function that will find the cod per muni
def cod(ratios): 
    median_ratio = ratios.median() 
    aad = (ratios - median_ratio).abs().mean() 
    return (aad / median_ratio) * 100

In [133]:
## group by neighborhood, apply the function
cod_by_nhood = valid_df.groupby('munidesc')['ratio'].apply(cod)

Blighted property defined by IAAO: A market area, neighborhood, or region may be defined as blighted if enough blighted properties exist to alter the functioning of the market due to declining economic conditions and social issues.

When considering COD, the IAAO treats distressed/blighted areas -- or places where "Arm’s Length transactions will have more dispersion in their sale ratios because the market is not as defined" -- differently than non-distressed/blighted areas. 

As a result, the COD standards for these areas covers a broader range (10 to 35, generally) than for others (5 to 15). 

In [136]:
cod_by_muni = cod_by_nhood.sort_values(ascending=False).reset_index()
cod_by_muni.tail(20)

,munidesc,ratio
101,Mt.Lebanon,13.193889
102,Bridgeville,13.106956
103,South Fayette,13.069790
104,Collier,13.057923
105,Shaler,13.050053
106,Scott,13.005122
107,West Deer,12.969229
108,Churchill,12.958736
109,Aspinwall,12.879701
110,Findlay,12.773134


In [137]:
## save to a CSV
cod_by_muni.to_csv('output/cod_by_muni.csv')

In [ ]:
# res_df = assessed_df[assessed_df['usedesc'].isin(['OTHER RESIDENTIAL STRUCTURE',
# 'MOBILE HOMES/TRAILER PKS',
# 'MOBILE HOME (IN PARK)',
# 'MOBILE HOME',
# 'CONDEMNED/BOARDED-UP',
# 'HUD PROJ #207/223',
# 'COMM APRTM CONDOS 5-19 UNITS',
# 'COMM APRTM CONDOS 40+ UNITS',
# 'COMM APRTM CONDOS 20-39 UNITS',
# 'HUD PROJ #236',
# 'NURSING HOME/PRIVATE HOS',
# 'HUD PROJ #232',
# 'HUD PROJ #213',
# 'HUD PROJ #202',
# 'INDEPENDENT LIVING (SENIORS)',
# 'TOWNHOUSE',
# 'OWNED BY METRO HOUSING AU',
# 'CONDO GARAGE UNITS',
# 'GROUP HOME',
# 'THREE FAMILY',
# 'APART: 5-19 UNITS',
# 'FOUR FAMILY',
# 'RES AUX BUILDING (NO HOUSE)',
# 'APART:20-39 UNITS',
# 'TWO FAMILY',
# 'ROWHOUSE',
# 'CONDOMINIUM COMMON PROPERTY',
# 'CONDO DEVELOPMENTAL LAND',
# 'CONDOMINIUM UNIT',
# 'CONDOMINIUM',
# 'APART:40+ UNITS',
# 'OFFICE/APARTMENTS OVER',
# 'SINGLE FAMILY'])]